## Pycytominer example pipeline
requires scipy=<1.7.3


### Set-up

In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, require, derived,
                         cellprofiler_results)
from utils.panels import save_panel

# Raw CellProfiler output for exp1_main. Resolves to data/cellprofiler_results/,
# which analysis/0_Download populates from S-BIAD2254.
CP_RESULTS = cellprofiler_results("exp1_main")

import os
import numpy as np
import pandas as pd
import string

# pycytominer
# from pycytominer import aggregate
from pycytominer import annotate
# from pycytominer import consensus
from pycytominer import feature_select
from pycytominer import normalize

import matplotlib.pyplot as plt
import seaborn as sns; sns.set_style("white")

# Set current working directory


In [ ]:
OutputDir = "results" # Where do you want to save the merged data csv? 
if not os.path.exists(OutputDir):
    os.makedirs(OutputDir)

cells = ['HCT116', 'HT29']

In [ ]:
## Figure settings 

dpi = 300
figsize = (2.24, 2.24)

plt.rcParams['pdf.fonttype'] = 42  
plt.rcParams['ps.fonttype'] = 42  


# Font sizes
title_size = 12
axis_label_size = 10
label_size = 6


### Perfrom quality control



In [ ]:
### Some settings

features = ['ImageQuality_PowerLogLogSlope_PHAandWGA', 'ImageQuality_PowerLogLogSlope_HOECHST', 'ImageQuality_PowerLogLogSlope_CONC','ImageQuality_PowerLogLogSlope_SYTO', 'ImageQuality_PowerLogLogSlope_MITO']
plates = ["PB000137", "PB000138", "PB000139", "PB000140", "PB000141", "PB000142"]

cutoff = 0.93

In [ ]:
# Initialize the dataframe
QC = pd.DataFrame()

for plate in plates:
    # Import QC for all cell lines 
    # Cluster-only tier: the BioImage Archive deposit carries
    # featICF_{nuclei,cells,cytoplasm} but not the per-plate QC tables or
    # featICF_spheroid, so this notebook cannot run from a download.
    QCfile = f"{CP_RESULTS}/spher-colo52/{plate}/QC/qcRAW_images.csv"

    # Import the data
    data = pd.read_csv(QCfile, index_col=0)
    data['barcode'] = plate

    data['flag'] = 0
    for feature in features: 
        data['flag'] = data['flag'] + (data[feature] > data[feature].quantile(cutoff)).astype(int)

    # Merge the data
    QC = pd.concat([QC, data])
    

# Update the source layout to match the new well assignments 
QC['well'] = QC['FileName_CONC'].str.split('-', expand=True)[1]
QC['well_number'] = QC['well'].str.split('(\d+)', expand=True)[1].str.lstrip('0').astype(int)
QC['well_letter'] = QC['well'].str.split('(\d+)', expand=True)[0].map(lambda x: ord(x) - 64)

QC['plate_well'] = QC['well'] + "_" + QC['barcode']

# Save the merged data
QC.to_csv("{}/QCFlags.csv".format(OutputDir), index=False)


In [ ]:
## QC

for plate in plates:

    # define a 386-well plate
    cols = 24
    rows = 16

    heatmap = np.zeros((rows,cols))

    plot_df = QC[QC['barcode'] == plate]

    for i in range(len(plot_df)):
        heatmap[plot_df['well_letter'].iloc[i]-1,plot_df['well_number'].iloc[i]-1] = plot_df['flag'].iloc[i]

    fig = plt.figure(figsize=figsize, dpi=dpi)
   
    %matplotlib inline
    yticklabels= list(string.ascii_uppercase)[:16]
    xticklabels = range(1, 25)

    ax = sns.heatmap(heatmap, linewidths=1, cmap='Reds', yticklabels=yticklabels, xticklabels=xticklabels, cbar=False)
    ax.set_title('# Flags: {}'.format(plate), fontsize=title_size)
    ax.xaxis.tick_top()
    # Hide major ticks but keep the labels
    ax.tick_params(axis='both', which='both', length=0)

    # We change the fontsize of minor ticks label 
    ax.tick_params(axis='both', labelsize=6)

    # box tight
    plt.tight_layout()

        # [not a paper panel] plt.savefig("{}/{}_QC.pdf".format(OutputDir, plate), dpi=dpi)
    pass
    plt.close()

    


### Preprocess with pycytominer

In [ ]:
## Import data and remove QC flags

data = pd.DataFrame()

for plate in plates:
    # Import QC for all cell lines 
    data_file = f"{CP_RESULTS}/spher-colo52/{plate}/results/featICF_spheroid.csv"

    # Import the data
    tmp= pd.read_csv(data_file, index_col=0)
    tmp['barcode'] = plate

    print("Data", plate ,":\n", tmp.shape)

    # Merge the data
    data = pd.concat([data, tmp])
    print("XXX", data.shape)

# Rename the plate_well using Metadata_barcode and well_id
data['plate_well'] = data['Metadata_Well'] + "_" + data['barcode']

df = data.merge(QC[['plate_well', 'well', 'flag']], left_on=['plate_well'], right_on=['plate_well'])

print("Data before QC:\n", df.shape)

# Filter out wells with 2 or more flags
df = df[df['flag'] < 2]

print("Data after QC:\n", df.shape, "\n")


## Are there any NaNs? 
nans = df.isna().sum()
nans = nans[nans > 0]

print("There are so many nans:\n", nans)


In [ ]:
# Remove non-data features from the list of features
ListOfFeatures = list(df.columns.values)
ListOfMetadata = list(df.columns[
    df.columns.str.contains("FileName_") |
    df.columns.str.contains("PathName_") |
    df.columns.str.contains("Metadata_")])
ListOfFeatures = list(set(ListOfFeatures) - set(ListOfMetadata) - set(['ObjectNumber', 'Number_Object_Number', 'well', 'flag']))

# ListOfMetadataNew = ["Metadata_Well", "cell_line"] 

# Remove all metadata, paths and filenames except for the well_id
df = df[ListOfFeatures +  ["Metadata_Well"]]

In [ ]:
# Import Metadata
dfLayout = pd.read_csv(metadata("spher_colo52-metadata.csv", "exp1_main"), sep=",")
# dfLayout = dfLayout.loc[~(dfLayout.layout_id == 'spher010-P1-L2')]
# Rename the plate_well using Metadata_barcode and well_id
dfLayout['plate_well'] = dfLayout['well_id'] + "_" + dfLayout['barcode']
dfLayout['name'] = dfLayout['cmpdname'].str[:5]

print(dfLayout.shape)
dfLayout.head()

In [ ]:
# Annotate: connect metadata to the feature data
# OBS: metadata will be prefixed with 'Metadata_'
annotated = annotate(df, platemap=dfLayout, join_on=['Metadata_plate_well', 'plate_well'],add_metadata_id_to_platemap=True, format_broad_cmap=False, clean_cellprofiler=False)
# annotated.to_csv("{}/annotated_data_{}.csv".format(OutputDir, cell_line))

annotated.shape

In [ ]:
# Remove non-data features from the list of features
ListOfFeatures = list(df.columns.values)
ListOfMetadata = list(df.columns[df.columns.str.contains("Metadata_")])
ListOfFeatures = list(set(ListOfFeatures) - set(ListOfMetadata) - set(["plate_well", "barcode", "cell_line"]))

In [ ]:
annotated['Metadata_norm_unit'] = annotated['Metadata_barcode'] + "_" + annotated['Metadata_cell_line']
units = annotated['Metadata_norm_unit'].unique()

In [ ]:
#
# Version 2: Normalize separately per 1) plate and 2) cell line
#

# # Normalize separately per cell line
# ListOfPlates = annotated['Metadata_layout_id'].unique()

# itnitialize an empty dataframe
normalized = pd.DataFrame(columns=annotated.columns.values)
normalized = normalized.drop(columns=['plate_well', 'barcode'])

for unit in units:
    
    annotated_temp = annotated[annotated['Metadata_norm_unit'] == unit]

    # Normalize: choose between standardize, robustize, mad_robustize, spherize 
    normalized_temp = normalize(annotated_temp, 
                                features=ListOfFeatures,image_features=False, 
                                meta_features="infer", samples="Metadata_pert_type == 'neg_con'", 
                                method="standardize")
    normalized = pd.concat([normalized, normalized_temp], ignore_index=True)

    print(unit)


In [ ]:
# Feature selection: "variance_threshold", "correlation_threshold", "drop_na_columns", "blocklist", "drop_outliers", "noise_removal",
to_clip_df = feature_select(normalized, features=ListOfFeatures, operation=["variance_threshold", "correlation_threshold","drop_na_columns", "blocklist" ])

In [ ]:
# Remove non-data features from the list
ListOfSelectedFeatures = list(to_clip_df.columns.values)
ListOfMetadata = list(to_clip_df.columns[to_clip_df.columns.str.contains("Metadata_")])
ListOfSelectedFeatures = list(set(ListOfSelectedFeatures) - set(ListOfMetadata))

In [ ]:
# Instead of removing the outliers, we can clip them to the 1st and 99th percentile.
selected_df = pd.concat([to_clip_df[ListOfMetadata], to_clip_df[ListOfSelectedFeatures].clip(lower=-40, upper=40, axis=1)], axis=1)
print(selected_df.shape)

In [ ]:
selected_df['Metadata_PlateWell'] = selected_df['Metadata_Well'] + "_" + selected_df['Metadata_barcode']
# Written to derived/, not to the old working tree's "../spher_colo52_v1/1_Data/
# results/", which does not exist here. derived/ also keeps a re-run from
# overwriting the deposited copy of these same tables.
for cell_line in ('HCT116', 'HT29'):
    out = derived('exp1_main', f'selected_data_MIP_{cell_line}.parquet')
    selected_df.query(f"Metadata_cell_line == '{cell_line}'").to_parquet(out)
    print(f'wrote {out}')